In [1]:
def identify_intent(question):
    q = question.strip().lower()

    if "දුම්රිය" in q or "කෝච්චි" in q:
        return "train_services"

    if "නවාතැන්" in q or "හෝටල්" in q:
        return "accommodation"

    if "කඳු නැග" in q or "තරණය" in q:
        return "hiking"

    if "පළාත" in q:
        return "province"

    if "දිස්ත්‍රික්ක" in q:
        return "district"

    if "උස" in q:
        if "සහ" in q or "අතරින්" in q:
            return "height_comparison"
        return "height"

    if "සුදුසු කාල" in q or "සුදුසු වේලාව" in q:
        return "visiting_time"

    return "unknown"

In [2]:
questions = [
    "ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?",
    "ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?",
    "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?",
    "රාවණා ඇල්ලේ උස කීයද?",
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
    "ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?",
    "පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?"
]

for question in questions:
    print("Question:", question)
    print("Detected intent:", identify_intent(question))
    print("-" * 50)

Question: ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?
Detected intent: province
--------------------------------------------------
Question: ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?
Detected intent: accommodation
--------------------------------------------------
Question: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?
Detected intent: hiking
--------------------------------------------------
Question: රාවණා ඇල්ලේ උස කීයද?
Detected intent: height
--------------------------------------------------
Question: රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?
Detected intent: height_comparison
--------------------------------------------------
Question: ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?
Detected intent: train_services
--------------------------------------------------
Question: පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?
Detected intent: visiting_time
--------------------------------------------------


In [3]:
entity_aliases = {
    "D001": ["ඇල්ල", "Ella"],

    "A001": [
        "පුංචි ශ්‍රී පාදය",
        "පුංචි සිරිපාදය",
        "Little Adam's Peak",
        "Mini Adam's Peak"
    ],

    "A002": ["ඇල්ල රොක්", "Ella Rock"],

    "A003": [
        "ආරුක්කු නවයේ පාලම",
        "Nine Arches Bridge"
    ],

    "A004": ["රාවණා ඇල්ල", "Ravana Falls"],

    "A005": ["රාවණා ගුහාව", "Ravana Cave"],

    "A006": ["දෝව රජමහා විහාරය"],

    "A007": [
        "දියලුම දිය ඇල්ල",
        "දියලුම ඇල්ල",
        "Diyaluma Falls"
    ],

    "A008": ["නිල්දිය පොකුණ"]
}

print("Entities configured:", len(entity_aliases))

Entities configured: 9


In [4]:
def identify_entities(question):
    text = question.casefold()

    candidates = []

    for entity_id, aliases in entity_aliases.items():
        for alias in aliases:
            alias_text = alias.casefold()

            start = text.find(alias_text)

            while start != -1:
                end = start + len(alias_text)

                candidates.append({
                    "id": entity_id,
                    "matched_text": question[start:end],
                    "start": start,
                    "end": end
                })

                start = text.find(alias_text, start + 1)

    # Match longer entity names first
    candidates.sort(
        key=lambda x: (
            -(x["end"] - x["start"]),
            x["start"]
        )
    )

    selected = []

    for candidate in candidates:
        overlaps = any(
            candidate["start"] < item["end"]
            and candidate["end"] > item["start"]
            for item in selected
        )

        if not overlaps:
            selected.append(candidate)

    selected.sort(key=lambda x: x["start"])

    return [
        {
            "id": item["id"],
            "matched_text": item["matched_text"]
        }
        for item in selected
    ]

In [5]:
questions = [
    "ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?",
    "ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?",
    "ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?",
    "රාවණා ඇල්ලේ උස කීයද?",
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?",
    "ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?",
    "පුංචි ශ්‍රී පාදයට යාමට සුදුසු කාලය කුමක්ද?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Intent:", identify_intent(question))
    print("Entities:", identify_entities(question))
    print("-" * 55)


Question: ඇල්ල පිහිටා තිබෙන්නේ කුමන පළාතේද?
Intent: province
Entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
-------------------------------------------------------

Question: ඇල්ලේ නවාතැන් පහසුකම් මොනවාද?
Intent: accommodation
Entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
-------------------------------------------------------

Question: ඇල්ලේ කඳු නැගීමට හැකි ස්ථාන මොනවාද?
Intent: hiking
Entities: [{'id': 'D001', 'matched_text': 'ඇල්ල'}]
-------------------------------------------------------

Question: රාවණා ඇල්ලේ උස කීයද?
Intent: height
Entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}]
-------------------------------------------------------

Question: රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?
Intent: height_comparison
Entities: [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}, {'id': 'A007', 'matched_text': 'දියලුම ඇල්ල'}]
-------------------------------------------------------

Question: ඇල්ලට පැමිණෙන දුම්රිය සේවා මොනවාද?
Intent: train_services
Entities: [{'i

Combine intent and entities

In [6]:
def analyze_question(question):
    return {
        "question": question,
        "intent": identify_intent(question),
        "entities": identify_entities(question)
    }


test = analyze_question(
    "රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?"
)

print(test)

{'question': 'රාවණා ඇල්ල සහ දියලුම ඇල්ල අතරින් උසින් වැඩි කුමක්ද?', 'intent': 'height_comparison', 'entities': [{'id': 'A004', 'matched_text': 'රාවණා ඇල්ල'}, {'id': 'A007', 'matched_text': 'දියලුම ඇල්ල'}]}
